<a id="inicio"></a>
<br><br><br>
<div align="left">
<h1><font size=6>Pandas</font></h1>
    <h2><font color="#5E5E5E" size=5>6. Operaciones básicas de preprocesamiento</font>  </h2>
<br><br><br><br>
</div>


<div align="right">
<font color="#B51700" size=4><b>Análisis y visualización de datos</b></font><br><br>
<font  size=3>Máster Universitario en Ingeniería Informática</font><br>
<font  size=3>Máster Universitario en Big Data y Computación en la Nube</font><br>
<font size=3>Máster Universitario en Inteligencia Artificial</font><br><br>
<font color="#5E5E5E" size=3><b>Luis de la Ossa  &nbsp;&nbsp;</b></font><span style="background-color:#B51700; color:white; padding:2px 6px; font-family:Arial, sans-serif;font-weight:bold;font-size:18px;">UCLM</span>
</div>

---


In [ ]:
import pandas as pd
import numpy as np

pd.__version__

A partir de los tipos y valores ausentes que ya sabemos identificar, se verá cómo descartar, rellenar o reemplazar datos, cómo definir un criterio de duplicación y cómo convertir tipos de forma explícita.

---
## <font color="#5E5E5E"> 1. Valores perdidos</font>

Es muy común que algunos valores no existan o no estén disponibles en los conjuntos de datos. En los archivos de datos representan los valores perdidos de manera arbitraria: espacio, `'?'`, `'NA'`, `'missing'`, etc. Sin embargo, en análisis de datos,  estos **valores** se denominan **perdidos** y su presencia requiere, en muchos casos, un análisis y tratamiento específico. Internamente, dentro del `DataFrame`, **los valores perdidos se deben representar con el valor especial *Not a Number** *(`NaN`, `pd.NA`, o `pd.NaT`). 

Por ejemplo, en el conjunto de datos `data/eukaryotes.tsv`, empleado en libretas anteriores, los valores perdidos se representan mediante el carácter `'-'`. 

In [ ]:
df_sequences = pd.read_csv('data/eukaryotes.tsv', sep='\t', index_col='Species')
df_sequences.tail()

Para estas columnas, el valor en algunas filas es el caracter `'-'`, que *no puede ser convertido a un número*. Debido a esto, la función `pd.read_csv()` determina que el contenido de la columna son cadenas de texto (`str`). Esto implica, por ejemplo, que el valor del número de genes de una de las filas con índice `Saccharomyces cerevisiae` se codifica como  `'155'` (`str`) y no `155` (`int`).

In [ ]:
df_sequences.info()

La función `pd.read_csv()` por defecto, asume que los valores perdidos se representan en el archivo de las formas más habituales (`""`, `NA`, `N/A`, `NaN`, etc.), aunque se puede especificar que ignore estas representaciones con `keep_default_na = False`. También admite un argumento denominado `na_values` que permite indicar cómo se representan los valores perdidos en el archivo, si esta representación es distinta de la habitual.


En este caso, al indicar que `na_values='-'`, la función de lectura sustituye el caracter `'-'` por el valor `NaN`, cuyo tipo es `float64`. Esto repercute también en el tipo de datos de las columnas afectadas, que ahora pasan a ser `float64`, ya que todos los valores existentes se pueden convertir a ese tipo de datos. También pueden observarse los cambios en la columna `Non-Null Count` que devuelve `DataFrame.info()`, y donde antes no se apreciaba la existencia de valores perdidos.

In [ ]:
df_sequences = pd.read_csv('data/eukaryotes.tsv', sep='\t',  na_values='-', index_col='Species')
df_sequences.tail()

In [ ]:
df_sequences.info()

Ahora, los valores para `Number of genes` y `Number of proteins` se codifican con números reales (`float64`) pero representan números enteros. Mediante el parámetro `dtype` se puede indicar que se utilice esta última representación. Sin embargo, a diferencia de lo que ocurre con `Publication year`, la especificación del tipo `int64` para estas columnas *generaría un error debido a la presencia de valores perdidos*. Cuando existen valores perdidos, se ha de utilizar el tipo `Int64`, que es similar a `int64`, pero sí acepta estos valores.  


<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> Cuando se utiliza `Int64` los valores perdidos representan con `<NA>`. `pd.NA` es un indicador de valor perdido que se pretende que sea consistente con todos los tipos de datos. Está todavía en fase experimental pero forma parte de varios tipos.
</div>

In [ ]:
new_types = {
    "Species": "string",
    "Kingdom": "string",
    "Class": "string",
    "Assembly status": "string",
    "Number of genes": "Int64",
    "Number of proteins": "Int64",
    "Publication year": "int64"
}

df_sequences = pd.read_csv('data/eukaryotes.tsv', sep='\t', dtype=new_types, na_values='-', index_col='Species')
df_sequences.tail()

### <font color="#5E5E5E"> Tratamiento de valores perdidos </font>

El método `Dataframe.dropna()`/`Series.dropna()` permite eliminar valores perdidos. El funcionamiento de `Series.dropna()` es sencillo, ya que simplemente elimina estos valores. El método `DataFrame.dropna()` en cambio admite varias opciones [**( + inf )**](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.dropna.html). Sus parámetros principales son:

* `axis`, que puede ser `0/index`, o `1/columns`; 
* `how`, que puede tomar los valores `any` o `all`. Descarta la fila/columna si, respectivamente, alguno o todos los valores son perdidos;
* `thresh`, que indica el número mínimo de valores conocidos necesarios para preservar una fila o columna.
* `subset`, que contiene el conjunto de filas/columnas a las que se aplica el método.

`how` y `thresh` no se pueden especificar simultáneamente. Además, este método también acepta el parámetro `inplace`.

In [ ]:
df_sequences.dropna()     # Por defecto, elimina todas las filas que incluyen valores perdidos. 
                          # Es equivalente a df.dropna(axis=0, any=true)
    
# Distintas posibilidades

#df_sequences.dropna(axis=1)               # Elimina las columnas que contienen valores perdidos. 
#df_sequences.dropna(axis=0, thresh=7)     # Elimina las filas que no contengan, al menos, siete valores. 
#df_sequences.dropna(axis=1, thresh=3000)  # Elimina las columnas que no contengan, al menos, 3000 valores. 
#df_sequences.dropna(how='any')            # Elimina las filas que contengan algún valor perdido
#df_sequences.dropna(axis=1, how='all')    # Elimina las columnas que tengan todos los valores perdidos

Otra posibilidad con respecto a los valores perdidos, consiste en sustituirlos mediante el método `DataFrame.fillna()`/`Series.fillna()`.
Admite varios parámetros:

* `axis`, (solo para *DataFrame*) que puede ser `0/index`, o `1/columns`. 
* `value`. Si es un escalar, sustituye todos los valores perdidos con ese valor. También permite especificar los valores utilizados en cada índice/columna mediante  
* `limit`. Si se especifica, solamente sustituye una cantidad limitada de los mismos, la especificada por este parámetro. Esto implica que algunos de éstos valores pueden __no__ ser imputados.

Este método también acepta el parámetro `inplace`.

In [ ]:
df_sequences.fillna(0)                                                 # Rellena todos los valores perdidos con el símbolo 0
#df_sequences.fillna({'Number of proteins':0,'Kingdom':'---'})           # Rellena cada columna con un valor

<div class="alert alert-block alert-warning">
    
<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
__Importante__: Es muy común sustituir los valores perdidos por valores arbitrarios como la media o mediana de la columna. Para esto se puede utilizar tambien `DataFrame.fillna()`. 
</div>

En `DataFrame.bfill()`/`Series.bfill()` y `DataFrame.ffill()`/`Series.ffill()` se usan para sustituir valores perdidos cuando los datos están ordenados.  `ffill()` propaga el último dato presente; `bfill()`, el siguient dato conocido. Ambos admiten el parámetro `limit`.

In [ ]:
df_sequences.ffill()   
df_sequences.ffill(limit = 2)
##df_sequences.bfill()

<div class="alert alert-block alert-warning">
    
<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
__Importante__: Hay que tener en cuenta que el valor utilizado puede provocar un cambio de tipo de datos en las columnas o filas. Estos métodos suelen ser de utilidad en series temporales, aunque en este caso se puede recurrir tambien a métodos de interpolación de distinta complejidad.
</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> 
Existe un método, denominado `DataFrame.combine_first()` que sustituye los `NaN` correspondientes al `DataFrame` original con los elementos correspondientes (alineando filas y columnas) en otro `DataFrame` que es pasado por argumento [**(+ inf)**](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.combine_first.html).

</div>

---
## <font color="#5E5E5E"> 2. Reemplazo de valores</font>


El método `Series.replace()` /`DataFrame.replace()` permite sustituir valores, de manera genérica, en una `Series` o un `DataFrame`. Propociona una ***gran cantidad de opciones*** [**( + inf )**](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.replace.html) que lo dotan de gran flexibilidad. Por ejemplo, puede trabajar con expresiones regulares. Puede ser utilizado también para la sustitución de valores perdidos.

In [ ]:
df_sequences.replace(to_replace=np.nan, value=0)                                               # Sustituye los valores perdidos por ceros.      
#df_sequences.replace(to_replace=np.nan, value={'Number of genes': 0, 'Assembly status':'----'}) # Sustituye los valores perdidos por ceros en una columna.

---
## <font color="#5E5E5E"> 3. Duplicados</font>

El método `Series.duplicated()` / `Dataframe.duplicated()` devuelve una serie de valores booleanos en la que se indica si cada elemento o fila está duplicado o no. Por otra parte, `Series.drop_duplicates()`/`DataFrame.drop_duplicates()` permite eliminar elementos/filas con valores duplicados. Estos métodos aceptan un parámetro, `keep`, que determina si conserva el primer elemento duplicado `'first'`, el último `'last'` o ninguno `False`. 


En el caso de `DataFrame.drop_duplicates()`, el método considera por defecto que dos filas están duplicadas si son iguales (salvo en el índice), pero recibe otro argumento, denominado `subset`, que permite determinar qué columnas se consideran en la comparación de dos filas.

In [ ]:
df_sequences.drop_duplicates(subset=['Kingdom', 'Class', 'Publication year']) # inplace = False

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> También existe el método `Index.drop_duplicates()` que devuelve un índice sin etiquetas repetidas. Por otra parte `DataFrame.set_flags(allows_duplicate_labels=False)` hace que se produzca una excepción al utilizar valores duplicados del índice.
</div>

---
## <font color="#5E5E5E"> 4. Cambios de tipos de datos</font>


Los métodos `DataFrame.astype(tipos)`/`Series.astype(tipo)` permiten cambiar el tipo de datos de las columnas del `DataFrame`, o de una `Series`. En el primer caso, el argumento `tipos` que toma el método es un `dtype` único para todo el `DataFrame` o un diccionario en el que las claves son las columnas afectadas, y el valor el nuevo tipo de datos. En el segundo, para las `Series`, `tipo` corresponde al nuevo tipo de datos. 

`DataFrame.astype()` devuelve un nuevo objeto `DataFrame`, pero no significa necesariamente que haga inmediatamente una copia física de todos sus datos


In [ ]:
df_sequences.astype({'Number of genes':'float64', 'Number of proteins':'Int64'}) # copy=True

<br>
<div align="right">
<a href="#inicio"><font size=5 color="#B51700"><i class="fa fa-arrow-circle-up" aria-hidden="true"></i></font></a>
</div>

---

<div align="right">
<font size=5 color="#B51700"><i class="fa fa-coffee" aria-hidden="true"></i></font>
</div>